# Table Playground

Interactive exploration of tabular results — render as Markdown,
LaTeX (booktabs), or as a bar chart for visual comparison.

**Compatible experiments:** `fronthaul_table`
(uses `kind == 'table'`).

In [ ]:
# Stage 12: shared setup — make `cordis` importable when this notebook
# is launched from notebooks/, then apply the IEEE paper rcParams.
import sys, logging
from pathlib import Path
from _playground_helpers import (
    setup_paper_style, load_latest_result, load_run, summarize,
)

import numpy as np
import matplotlib.pyplot as plt

# Set use_latex=False if pdflatex isn't on PATH (e.g. on a compute node).
setup_paper_style(use_latex=True)

logging.basicConfig(level=logging.WARNING, format='%(levelname)-7s %(message)s')

In [ ]:
EXPERIMENT = 'fronthaul_table'

result = load_latest_result(EXPERIMENT)
assert result.kind == 'table'
summarize(result)
print()
print('Table keys:', list(result.table_data.keys()))

## 1. Markdown render (good for previews + GitHub READMEs)

In [ ]:
from cordis.plotting import to_markdown_table

print(to_markdown_table(result.table_data))

## 2. LaTeX render for the paper

Uses `booktabs` style.  Copy-paste the output into your paper's `.tex`
(or use `\input{...}` from a saved `.tex` file).

In [ ]:
from cordis.plotting import to_latex_table

tex = to_latex_table(
    result.table_data,
    caption='Fronthaul scalars per algorithm (one Monte Carlo realization).',
    label='tab:fronthaul',
)
print(tex)

# Save it for \input{} from your paper.
out = Path('../figures/playground/fronthaul_table.tex')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(tex)
print(f'wrote {out}')

## 3. Bar chart — visual comparison

When numbers span orders of magnitude (e.g. fronthaul scalars from 4
to 280), a log-y bar chart often communicates the gap better than the
numeric table alone.

In [ ]:
from cordis.plotting import bar_chart, figsize

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
bar_chart(
    result.table_data,
    column='real_scalars',
    ax=ax,
    ylabel='real scalars per AP per iter',
)
ax.set_yscale('log')
ax.set_title('Fronthaul cost (log scale)')
plt.tight_layout()
plt.show()

## 4. Highlight CORDIS rows in the LaTeX output

For paper figures it's nice to bold-face the proposed algorithms
so the reader's eye lands on them.  This is a post-process on the
`to_latex_table` output.

In [ ]:
tex_with_highlight = tex
for alg in ('CORDIS-Split', 'CORDIS-ADMM'):
    tex_with_highlight = tex_with_highlight.replace(
        f'{alg}', f'\\textbf{{{alg}}}'
    )
print(tex_with_highlight)

## 5. Figsize variants

In [ ]:
# Figsize variants — `figsize` returns (w, h) in inches for matplotlib.
# width: 'single' (one column), 'double' (two-column), 'third' (3-up panel).
# aspect: w/h ratio.  Tweak both to fit your paper layout.
from cordis.plotting import figsize

for width in ('single', 'double', 'third'):
    w, h = figsize(width=width, aspect=3/2)
    print(f'{width:>6}: ({w:.2f}, {h:.2f}) inches')

# Example: tight three-up panel for a paper sub-figure
# fig, axes = plt.subplots(1, 3, figsize=figsize(width='double', aspect=3.5/1.5))

## 6. Save the bar chart with provenance

In [ ]:
# Save with provenance metadata (Git SHA, creation date, etc. — embedded
# into the PDF's metadata, prepended as comments in the .pgf).
from cordis.plotting import save_paper_figure

# Adjust EXPERIMENT and metric labels to match the figure above.
out = save_paper_figure(
    fig,
    base_path=f'../figures/playground/{EXPERIMENT}_demo',
    formats=('pdf', 'png'),       # add 'pgf' on systems with LaTeX
    metadata={'Experiment': EXPERIMENT, 'Notebook': 'playground'},
)
for p in out:
    print('wrote', p)